# End of week 1 exercise

The purpose of this tool is to build a recipes book using some reference websites as an input.

### Setup

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from urllib.parse import urljoin, urldefrag, urlparse
from openai import OpenAI
from recipe_scrapers import scrape_me


In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL_GPT = 'gpt-5-nano'
openai = OpenAI()

MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = 'http://localhost:11434'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
# set up environment

recipe_reference_sites = [
    "https://www.101cookbooks.com",
    "https://www.bbcgoodfood.com/recipes/collection/one-pot-recipes",
]


### Functions & prompts to fetch relevant recipes

In [ ]:
filter_system_prompt = """
    You are given a list of URLs found on a recipe website.
    Your task is to identify which URLs point directly to individual recipe pages.

    Include:
    - URLs for specific recipes
    - URLs that appear to contain one complete recipe

    Exclude:
    - category pages
    - collection pages
    - homepage links
    - search pages
    - author pages
    - tag pages
    - shopping pages
    - social media links
    - newsletter or account links
    - generic article pages that are not recipes
    
    Return only valid JSON in this exact structure:
    {
        "links": [
            {
                "recipe_name": "Name of the recipe if you can infer it from the URL, otherwise empty string",
                "url": "https://example.com/full-recipe-url"
            }
        ]
    }
}
"""

In [ ]:
def get_links_user_prompt(candidate_urls):
    user_prompt = f"""
        Here is a list of normalized URLs found on a website.
        Select only the URLs that point directly to individual recipe pages.
        
        URLs:
        {candidate_urls}
    """
    return user_prompt

In [ ]:
def normalize_links(base_url, raw_links):
    normalized_links = []
    for link in raw_links:
        absolute_url = urljoin(base_url, link)
        absolute_url, _ = urldefrag(absolute_url)
        parsed = urlparse(absolute_url)
        if parsed.scheme not in ("http", "https"):
            continue
        normalized_links.append(absolute_url)
    return sorted(set(normalized_links))


In [ ]:
def select_relevant_recipe_links(sites_to_scrape):
    relevant_links = {"links": []}
    for site in sites_to_scrape:
        print(f"Fetching links from {site}")
        raw_urls = fetch_website_links(site)
        candidate_urls = normalize_links(site, raw_urls)
        print(f"Found {len(raw_urls)} raw links")
        print(f"Found {len(candidate_urls)} normalized unique links to filter")

        response = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=[
                {"role": "system", "content": filter_system_prompt},
                {"role": "user", "content": get_links_user_prompt(candidate_urls)}
            ],
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
        recipe_links = result.get("links", [])
        print(f"Found {len(recipe_links)} relevant recipes")
        relevant_links["links"].extend(recipe_links)
        
    return relevant_links


In [ ]:
def scrape_recipe(recipe_link):
    url = recipe_link["url"]
    try:
        scraper = scrape_me(url)
        return {
            "recipe_name": scraper.title(),
            "url": url,
            "description": scraper.description(),
            "ingredients": scraper.ingredients(),
            "instructions": scraper.instructions_list(),
            "total_time": scraper.total_time(),
            "yields": scraper.yields(),
            "image": scraper.image(),
        }
    except Exception as e:
        print(f"Could not scrape {url}: {e}")
        return {
            "recipe_name": recipe_link.get("recipe_name", ""),
            "url": url,
            "error": str(e),
        }


### Functions & prompts to build recipes book

In [ ]:
recipe_book_system_prompt = """
    You are a helpful cooking assistant.
    You will receive structured recipe data scraped from recipe websites.
    Your task is to create a clean, readable recipe book in Markdown.
    
    Rules:
    - Do not invent ingredients, instructions, timings, or servings.
    - Use only the data provided.
    - If a field is missing, omit that section.
    - Keep each recipe concise and easy to scan.
    - Include the source URL for each recipe.
"""


In [ ]:
def get_recipe_book_user_prompt(recipe_book):
    return f"""
        Create a Markdown recipe book from this structured recipe data:
        {json.dumps(recipe_book, indent=2)}
    """


In [ ]:
def build_recipe_book(recipe_links, limit=10):
    recipe_book = []
    failed_recipes = []
    for recipe_link in recipe_links[:limit]:
        recipe = scrape_recipe(recipe_link)
        if "error" in recipe:
            failed_recipes.append(recipe)
        else:
            recipe_book.append(recipe)
        print(f"Processed: {recipe.get('recipe_name')}")
    return recipe_book, failed_recipes


In [ ]:
def create_markdown_recipe_book(recipe_book):
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": recipe_book_system_prompt},
            {"role": "user", "content": get_recipe_book_user_prompt(recipe_book)}
        ]
    )
    return response.choices[0].message.content


In [ ]:
def save_recipe_book(recipe_book, markdown_recipe_book):
    with open("recipe_book.json", "w", encoding="utf-8") as f:
        json.dump(recipe_book, f, indent=2, ensure_ascii=False)

    with open("recipe_book.md", "w", encoding="utf-8") as f:
        f.write(markdown_recipe_book)

    print("Saved recipe_book.json")
    print("Saved recipe_book.md")

### Workflow

In [ ]:
recipes = select_relevant_recipe_links(recipe_reference_sites)

recipe_book, failed_recipes = build_recipe_book(recipes["links"], limit=10)
print(f"Successfully scraped: {len(recipe_book)}")
print(f"Failed: {len(failed_recipes)}")

markdown_recipe_book = create_markdown_recipe_book(recipe_book)
save_recipe_book(recipe_book, markdown_recipe_book)
